In [1]:
## first create image dir of only healthy eyes, (use healthy_annotations.csv)
import sys
import importlib
sys.path.append('/content/drive/MyDrive/Colab Notebooks/')
import preprocessing
importlib.reload(preprocessing)
from preprocessing import preprocessing
import RetinaDataset
importlib.reload(RetinaDataset)
from RetinaDataset import RetinaDataset
import pandas as pd



In [2]:
## first only train on healthy individuals to see if it works
healthy_annotations_path = '/content/drive/MyDrive/Colab Notebooks/healthy_annotations.csv'
og_img_dir = '/content/drive/MyDrive/Colab Notebooks/Images'
processed_img_dir = '/content/drive/MyDrive/Colab Notebooks/preprocessed'


In [3]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
import torch

torch.manual_seed(42)  # For reproducibility

def z_normalize(image):
    # Assuming image is a PyTorch tensor of shape (C, H, W)
    # Create a mask for non-black pixels
    mask = image > 0
    for c in range(3):  # Loop over channels
        channel = image[c]
        # Mask out black pixels
        valid_pixels = channel[mask[c]]

        # Calculate mean and std of non-black pixels
        mean = valid_pixels.mean()
        std = valid_pixels.std()

        # Normalize the non-black pixels
        image[c] = torch.where(mask[c], (channel - mean) / (std + 1e-5), channel)
    return image

class TransformedRetinaDataset(RetinaDataset):
    def __init__(self, annotations_file, img_dir, transform=None):
        super().__init__(annotations_file, img_dir)
        self.transform = transform

    def __getitem__(self, idx):
        # Get the sample using the parent class method
        sample = super().__getitem__(idx)
        image, age = sample['image'], sample['age']

        # print("This is type of image after getting it from RetinaDataset:_ ", type(image))

        # Apply the transformations
        if self.transform:
            image = self.transform(image)

        # After other transformations, apply z-normalization
        image = z_normalize(image)


        return {'image': image, 'age': age}

original = transforms.Compose([
    transforms.ToTensor()
])


translate_transform1 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
])

flip_and_color_jitter_transform1 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

translate_transform2 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2)),
    transforms.ToTensor(),
])

flip_and_color_jitter_transform2 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.ToTensor(),
])


# Create the original dataset
# original_dataset = RetinaDataset(annotations_file=healthy_annotations_path, img_dir=processed_img_dir)

# Create augmented datasets with different transformations
datasets = []
# transformations = [original, rotate_transform, translate_transform, flip_and_color_jitter_transform]
transformations = [original,translate_transform1 , flip_and_color_jitter_transform1, translate_transform2, flip_and_color_jitter_transform2] # only use original dataset to check

for transform in transformations:
    datasets.append(TransformedRetinaDataset(annotations_file=healthy_annotations_path, img_dir=processed_img_dir, transform=transform))

# Concatenate all datasets
combined_dataset = ConcatDataset(datasets)


# Calculate the lengths of each split
train_length = int(0.8 * len(combined_dataset))
val_length = int(0.1 * len(combined_dataset))
test_length = len(combined_dataset) - train_length - val_length

# Perform the split
train_dataset, val_test_dataset = random_split(combined_dataset, [train_length, val_length + test_length])
val_dataset, test_dataset = random_split(val_test_dataset, [val_length, test_length])

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [4]:
### model, loss function and optimizer definitions

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import resnet101, ResNet101_Weights

from torch.optim.lr_scheduler import ReduceLROnPlateau

# load pretrained ResNet model with the initialized weights
# model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
model = models.resnet101(weights=ResNet101_Weights.DEFAULT)


# Freeze all layers initially
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1)  # modify it to make a regression at the last layer (age predicition)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print('Working on: ', device)


# Define the loss function and optimizer
criterion = nn.MSELoss()  # Mean Squared Error Loss
# criterion = nn.SmoothL1Loss() # used in the similar study
optimizer = optim.Adam(model.parameters(), lr = 1e-3, betas=(0.9, 0.999))
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=5, verbose=True) # reduce lr when validation loss start to plateau


def train_model(model, train_loader, val_loader, criterion, optimizer,scheduler,  num_epochs=10, early_stopping_patience=10):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    for epoch in range(num_epochs):
        # training loop
        model.train()
        train_loss = 0.0
        train_mae = 0.0

        for batch in train_loader:
            inputs = batch['image'].to(device)
            labels = batch['age'].to(device)

            optimizer.zero_grad()  # Zero the parameter gradients
            outputs = model(inputs)
            loss = criterion(outputs, labels.unsqueeze(1))
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_mae += torch.abs(outputs - labels.unsqueeze(1)).mean().item()

        train_loss = train_loss / len(train_loader)
        train_mae = train_mae / len(train_loader)

        # validation loop
        model.eval()
        val_loss = 0.0
        val_mae = 0.0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['image'].to(device)
                labels = batch['age'].to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels.unsqueeze(1))

                val_loss += loss.item()
                val_mae += torch.abs(outputs - labels.unsqueeze(1)).mean().item()

        val_loss = val_loss / len(val_loader)
        val_mae = val_mae / len(val_loader)
        print(f'Epoch {epoch+1}/{num_epochs} - Training loss: {train_loss:.4f}, Training MAE: {train_mae:.4f}', ', ',  f'Validation loss: {val_loss:.4f}, Validation MAE: {val_mae:.4f}')

         # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
          epochs_no_improve += 1
          if epochs_no_improve == early_stopping_patience:
              print("Early stopping triggered")
              break

        # learning rate sheduler
        scheduler.step(val_loss)

train_model(model, train_loader, val_loader, criterion, optimizer,scheduler=scheduler, num_epochs=50)

Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth
100%|██████████| 171M/171M [00:00<00:00, 188MB/s]


Working on:  cuda
Epoch 1/50 - Training loss: 1004.3004, Training MAE: 26.5930 ,  Validation loss: 252.3694, Validation MAE: 12.7756
Epoch 2/50 - Training loss: 235.2332, Training MAE: 12.0893 ,  Validation loss: 214.2929, Validation MAE: 11.5250
Epoch 3/50 - Training loss: 219.8563, Training MAE: 11.7174 ,  Validation loss: 215.7133, Validation MAE: 11.6963
Epoch 4/50 - Training loss: 216.6219, Training MAE: 11.6251 ,  Validation loss: 211.0864, Validation MAE: 11.4552
Epoch 5/50 - Training loss: 203.1319, Training MAE: 11.2530 ,  Validation loss: 198.3120, Validation MAE: 11.1774
Epoch 6/50 - Training loss: 199.0817, Training MAE: 11.1266 ,  Validation loss: 188.7923, Validation MAE: 10.9109
Epoch 7/50 - Training loss: 192.6609, Training MAE: 10.9325 ,  Validation loss: 181.5173, Validation MAE: 10.7287
Epoch 8/50 - Training loss: 185.0154, Training MAE: 10.7357 ,  Validation loss: 179.0592, Validation MAE: 10.6905
Epoch 9/50 - Training loss: 176.9675, Training MAE: 10.4986 ,  Valida

In [5]:
### eval on test set
def test_model(model, test_loader, criterion):
    model.eval()

    test_loss = 0.0
    test_mae = 0.0
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['image'].to(device)
            labels = batch['age'].to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels.unsqueeze(1))

            test_loss += loss.item()
            test_mae += torch.abs(outputs - labels.unsqueeze(1)).mean().item()

    test_loss /= len(test_loader)
    test_mae /= len(test_loader)

    print(f'Test Loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}')

test_model(model, test_loader, criterion)


Test Loss: 112.8366, Test MAE: 8.4894


In [7]:

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# Define a new optimizer that includes all parameters (potentially with different learning rates)
optimizer = optim.Adam([
    {'params': model.fc.parameters()},  # Optionally use a higher lr for the newly unfrozen layers
    {'params': model.layer4.parameters(), 'lr': 1e-4},  # Example of setting a specific layer's lr
], lr=1e-4)

train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=scheduler,num_epochs=200, early_stopping_patience=10)
test_model(model,test_loader, criterion)
model_path = 'resnet101_2.pth'
torch.save(model.state_dict(), model_path)

Epoch 1/200 - Training loss: 88.4539, Training MAE: 7.3220 ,  Validation loss: 70.8488, Validation MAE: 6.6890
Epoch 2/200 - Training loss: 50.5400, Training MAE: 5.4919 ,  Validation loss: 45.4281, Validation MAE: 5.1717
Epoch 3/200 - Training loss: 33.4362, Training MAE: 4.4536 ,  Validation loss: 35.5536, Validation MAE: 4.5897
Epoch 4/200 - Training loss: 27.0272, Training MAE: 4.0260 ,  Validation loss: 31.5343, Validation MAE: 4.2882
Epoch 5/200 - Training loss: 22.0090, Training MAE: 3.6283 ,  Validation loss: 27.2462, Validation MAE: 3.9632
Epoch 6/200 - Training loss: 21.1911, Training MAE: 3.5653 ,  Validation loss: 26.4777, Validation MAE: 3.8939
Epoch 7/200 - Training loss: 18.2805, Training MAE: 3.3273 ,  Validation loss: 21.6376, Validation MAE: 3.4602
Epoch 8/200 - Training loss: 16.1629, Training MAE: 3.1276 ,  Validation loss: 20.4503, Validation MAE: 3.4585
Epoch 9/200 - Training loss: 14.6539, Training MAE: 2.9745 ,  Validation loss: 24.8804, Validation MAE: 3.5325
E

In [ ]:
'''
possible steps for improving performance/finetuning:

- experiment with diff learning rates (e.g. 1e-3, 1e-4, 1e-5)
- try different batch sizes (e.g. 8, 16, 32, 64)
- maybe try again to use ResNet101 because maybe ResNet152 is too complex for the small dataset we have (or maybe even ResNet50???)
- try different optimizers (e.g. SGD, AdamW, RMSPROP)
- adjust learning rate scheduler params like patience and factor
- try to add dropout layers (not sure if possible with pretrained model), values typically range from 0.2 to 0.5
- data augmentation (synthetically alter number of data points with young age, because we have way more old individuals than young)
- ...
'''